### The Search For The Model
Looking for the best assortment of features that give us the best model (baseline) -> make sure to use 1994 train, 1995 val, and 96 last test


# *  import libraries

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time, math
import os
import re

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer # applies dif preprocessing operations to dif columns in dataset
from sklearn.impute import SimpleImputer # fills in missing values in dataset using strats like mean, median, or most frequent
from sklearn.pipeline import Pipeline # chains multiple preprocessing steps and models together into a single object
from sklearn.preprocessing import OneHotEncoder, StandardScaler # converts categorical variables to seperate binary / scaler: standardizes numerical features so they have a mean of 0 and std of 1
from sklearn.linear_model import LogisticRegression # lets u create a logistic regression model for binary classification tasks
from sklearn.dummy import DummyClassifier # create a baseline model that makes predictions based on simple rules or random chance
from sklearn.metrics import accuracy_score, precision_score, recall_score, average_precision_score, f1_score, confusion_matrix
# just dif ways to measure how well a classification model is performing

# Load and Clean Really Quick

In [6]:
df = pd.read_csv('../data/fraud_oracle.csv') # since they are in the both folder, we need to go one level up to access the data folder and then access the csv file

df = df[df['MonthClaimed'] != '0'].copy()
assert len(df) == 15419

# dealing with Age
df['AgeMissing'] = (df['Age'] == 0).astype(int) # create a new column called AgeMissing that is 1 if Age is 0 and 0 otherwise
df['Age'] = df['Age'].replace(0, np.nan) # replace 0s in Age with NaN values

In [10]:
# check cell
assert len(df) == 15419, f"Expected 15,419 rows after dropping the corrupt row, got {len(df)}"
assert (df['MonthClaimed'] != '0').all() and (df['DayOfWeekClaimed'] != '0').all(), "A '0' claim date is still present"
assert df['AgeMissing'].sum() == 319, f"Expected 319 unknown ages, got {df['AgeMissing'].sum()}"
assert df.loc[df['AgeMissing'] == 1, 'Age'].isna().all(), "Unknown ages weren't set to NaN"
assert df['Age'].dropna().min() > 0, "An Age of 0 is still in the data"
print(f"✓ Step 1 | fraud cases: {df['FraudFound_P'].sum()} ({df['FraudFound_P'].mean():.2%})")

✓ Step 1 | fraud cases: 923 (5.99%)


one agreed column list (straight from the data dictionary keep/drop)

In [7]:
numeric_cols = ['Age', 'Deductible', 'VehiclePrice', 'Days_Policy_Accident',
                'Days_Policy_Claim', 'PastNumberOfClaims', 'AgeOfVehicle',
                'AgeOfPolicyHolder', 'NumberOfCars', 'AgeMissing']
categorical_cols = ['Month', 'WeekOfMonth', 'DayOfWeek', 'Make', 'AccidentArea',
                    'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed',
                    'Sex', 'MaritalStatus', 'Fault', 'VehicleCategory',
                    'DriverRating', 'AgentType', 'BasePolicy']
# Dropped (with reasons in a markdown cell): PolicyNumber, RepNumber, PolicyType,
# PoliceReportFiled, WitnessPresent, NumberOfSuppliments, AddressChange_Claim.
# Year is used only for splitting.


In [11]:
# check cell 2
leaky_or_dropped = {'PolicyNumber', 'RepNumber', 'PolicyType', 'PoliceReportFiled',
                    'WitnessPresent', 'NumberOfSuppliments', 'AddressChange_Claim'}
features = numeric_cols + categorical_cols
assert not set(numeric_cols) & set(categorical_cols), "A column is listed as both numeric and categorical"
assert not set(features) & leaky_or_dropped, f"Dropped column still in features: {set(features) & leaky_or_dropped}"
assert 'FraudFound_P' not in features and 'Year' not in features, "Label or Year is in the features"
assert set(features) <= set(df.columns), f"Missing columns: {set(features) - set(df.columns)}"
assert len(features) == len(set(features)), "Duplicate column in features"
assert 'Fault' in features and 'AgeOfPolicyHolder' in features, "Fault / AgeOfPolicyHolder should be kept (per feedback)"
print(f"✓ Step 2 | {len(numeric_cols)} numeric + {len(categorical_cols)} categorical = {len(features)} features")

✓ Step 2 | 10 numeric + 15 categorical = 25 features


# Reasons Features Dropped
* Policy Number
* Rep Number
* Policy Type
* Police Report Filed - known only after claim is sent
* Witness Present
* Number of Suppliments
* Address Change Claim
* Year - used only for splitting

Converting text ranges to numbers - Mukundan

In [12]:
def parse_numeric_category(value, column):
  if pd.isna(value):
    return np.nan, "missing"

  s = str(value).strip().lower()

  # Special categories
  if s == "none":
    return 0.0, "none"

  if s == "new":
    return 0.0, "new"

  if s == "no change":
    return np.nan, "no_change"

  if s == "under 6 months":
    return 0.25, "under_6_months"

  # Ranges: "26 to 30", "3 to 4 vehicles", etc.
  found_match = re.fullmatch(
    r"(\d+(?:\.\d+)?)\s*to\s*(\d+(?:\.\d+)?)(?:\s+\w+)?",
    s
  )
  if found_match:
    low, high = map(float, found_match.groups())
    return (low + high) / 2, "range"

  # Open-ended values
  found_match = re.fullmatch(
    r"(more than|less than|over)\s+(\d+(?:\.\d+)?)(?:\s+\w+)?",
    s
  )
  if found_match:
    direction, number = found_match.groups()
    number = float(number)
    if direction in ("more than", "over"):
      return number + 1, direction.replace(" ", "_")
    else:  # "less than"
      return number - 1, direction.replace(" ", "_")
    
  # Exact values with optional units
  found_match = re.fullmatch(
    r"(\d+(?:\.\d+)?)(?:\s+\w+)?",
    s
  )
  if found_match:
    return float(found_match.group(1)), "exact"

  raise ValueError(
    f"Unrecognized value {value!r} in column {column!r}"
  )



range_cols = ['VehiclePrice', 'Days_Policy_Accident', 'Days_Policy_Claim',
              'PastNumberOfClaims', 'AgeOfVehicle', 'AgeOfPolicyHolder', 'NumberOfCars']

# Run BEFORE the conversion loop
n_categories_before = df[range_cols].nunique()

for col in range_cols:
    df[col] = df[col].apply(lambda v: parse_numeric_category(v, col)[0])

In [13]:
# step cell 3 check
n_categories_after = df[range_cols].nunique()
merged = n_categories_before[n_categories_before != n_categories_after]
assert merged.empty, f"Categories collapsed into the same number: {merged.to_dict()}"
assert df[range_cols].notna().all().all(), f"NaNs after conversion: {df[range_cols].isna().sum()[lambda s: s > 0].to_dict()}"
assert all(pd.api.types.is_numeric_dtype(df[c]) for c in range_cols), "A range column isn't numeric"
print("✓ Step 3 | no merged categories, no NaNs, all numeric")

✓ Step 3 | no merged categories, no NaNs, all numeric


# Split by Year, then group rare makes using 1994 only

In [14]:
train_df = df[df['Year'] == 1994].copy()
val_df = df[df['Year'] == 1995].copy()
test_df = df[df['Year'] == 1996].copy() # Dont touch

common = train_df['Make'].value_counts()
common = common[common >= 50].index
for d in (train_df, val_df, test_df):
    d['Make'] = d['Make'].where(d['Make'].isin(common), 'Other')

features = numeric_cols + categorical_cols
X_train, y_train = train_df[features], train_df['FraudFound_P']
X_val, y_val = val_df[features], val_df['FraudFound_P']


In [15]:
# step 4 check
assert set(train_df['Year']) == {1994} and set(val_df['Year']) == {1995} and set(test_df['Year']) == {1996}, "A split has the wrong year"
assert len(train_df) + len(val_df) + len(test_df) == len(df), "Rows lost or duplicated in the split"
assert set(train_df.index).isdisjoint(val_df.index) and set(val_df.index).isdisjoint(test_df.index), "Splits overlap"
assert set(val_df['Make']) <= set(train_df['Make']), "1995 has a car make that 1994 never saw"
assert list(X_train.columns) == list(X_val.columns) == features, "Train/val columns don't match the feature list"

# The rare-make threshold must come from 1994 only
recount = train_df['Make'].value_counts()
assert (recount.drop('Other', errors='ignore') >= 50).all(), "A rare make wasn't grouped into 'Other'"

TEST_SCORED = False   # flipped to True only in Step 9
print(pd.DataFrame({'claims': [len(train_df), len(val_df), len(test_df)],
                    'fraud rate': [d['FraudFound_P'].mean() for d in (train_df, val_df, test_df)]},
                   index=[1994, 1995, 1996]).round(4))
print("✓ Step 4")

      claims  fraud rate
1994    6141      0.0666
1995    5195      0.0579
1996    4083      0.0522
✓ Step 4


# Logistic Regression in a Pipeline

In [16]:
preprocess = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
])

logreg = Pipeline([('prep', preprocess),
                   ('model', LogisticRegression(class_weight = 'balanced', max_iter=3000, random_state=42))])
logreg.fit(X_train, y_train)
lr_scores = logreg.predict_proba(X_val)[:, 1]

In [17]:
# step 5 model check
age_imputer = logreg.named_steps['prep'].named_transformers_['num'].named_steps['impute']
age_pos = numeric_cols.index('Age')
assert np.isclose(age_imputer.statistics_[age_pos], X_train['Age'].median()), "Age median wasn't learned from 1994 only"

scaler = logreg.named_steps['prep'].named_transformers_['num'].named_steps['scale']
assert scaler.n_samples_seen_ == len(X_train), "Scaler saw more rows than the 1994 training set"

assert len(lr_scores) == len(y_val), "Wrong number of validation scores"
assert np.isfinite(lr_scores).all() and ((lr_scores >= 0) & (lr_scores <= 1)).all(), "Scores aren't valid probabilities"
assert lr_scores.std() > 0, "Model gives every claim the same score"
print(f"✓ Step 5 | {len(logreg[:-1].get_feature_names_out())} features after one-hot encoding")

✓ Step 5 | 89 features after one-hot encoding


# Step 6: One Metrics Function For Every Approach

In [ ]:
# this method is for evaluating the model performance based on the scores and the true labels
def evaluate(name, y, scores):
    y = np.asarray(y)
    order = np.argsort(-scores, kind='stable')
    row = {'Approach': name, 'PR-AUC': average_precision_score(y, scores)}
    for frac in (0.10, 0.20):
        top = y[order[:int(np.ceil(frac*len(y)))]]
        row[f'Fraud caught @ top {int(frac*100)}%'] = top.sum() / y.sum()
        if frac == 0.20:
            p,r = top.mean(), top.sum() / y.sum()
            row['Precision @ top 20%'] = p
            row['F1 @ top 20%'] = 2*p*r/(p+r) if p+r > 0 else 0
    return row


In [21]:
# step 6 check
y_toy = np.array([1]*10 + [0]*90)          # 10% fraud
perfect = evaluate('perfect', y_toy, np.linspace(1, 0, 100))   # fraud ranked first
worst   = evaluate('worst',   y_toy, np.linspace(0, 1, 100))   # fraud ranked last

assert np.isclose(perfect['PR-AUC'], 1.0)
assert perfect['Fraud caught @ top 10%'] == 1.0 and perfect['Fraud caught @ top 20%'] == 1.0
assert np.isclose(perfect['Precision @ top 20%'], 0.5)    # 10 fraud out of 20 flagged
assert worst['Fraud caught @ top 10%'] == 0 and worst['Fraud caught @ top 20%'] == 0
print("✓ Step 6 | evaluate() gives the right answers on known cases")

✓ Step 6 | evaluate() gives the right answers on known cases


# step 7: building the 4 approach table on 1995

In [23]:
rng = np.random.default_rng(42) # seeded random number generator for reproducibility
random_scores = rng.random(len(y_val)) # generate random scores for each validation sample

# lookup rule 1994 fraud rate for each base policy x fauly group
rates = train_df.groupby(['BasePolicy', 'Fault'])['FraudFound_P'].mean() # group by base policy and fault, then calculate the mean fraud rate for each group
base_rate = y_train.mean() # overall fraud rate in the training set
lookup_scores = np.array([rates.get(k,base_rate) # get the fraud rate for each base policy x fault group in the validation set, or use the overall base rate if the group wasn't seen in training
                         for k in zip(val_df['BasePolicy'], val_df['Fault'])])

results = pd.DataFrame([ # evaluate() returns a dict of metrics for each approach, which we collect into a DataFrame
    evaluate('Random', y_val, random_scores),
    evaluate('Lookup: coverage + fault', y_val, lookup_scores),
    evaluate('Logistic regression', y_val, lr_scores)
])


In [24]:
# step 7 check:
# Random picking should catch about 10% / 20%. Averaging many seeds rules out a lucky draw.
rand_rows = [evaluate('r', y_val, np.random.default_rng(s).random(len(y_val))) for s in range(50)]
rand_10 = np.mean([r['Fraud caught @ top 10%'] for r in rand_rows])
rand_20 = np.mean([r['Fraud caught @ top 20%'] for r in rand_rows])
assert abs(rand_10 - 0.10) < 0.02 and abs(rand_20 - 0.20) < 0.02, f"Random baseline is off: {rand_10:.3f}, {rand_20:.3f}"

# Lookup rule: built from 1994 only, at most 3 coverage types x 2 fault values
assert len(rates) <= 6, "Lookup table has unexpected groups"
assert np.isclose(rates.mean(), train_df.groupby(['BasePolicy', 'Fault'])['FraudFound_P'].mean().mean()), "Lookup rates didn't come from 1994"
assert not np.isnan(lookup_scores).any(), "Some 1995 claims got no lookup score"

# "Always say not fraud": PR-AUC should equal the 1995 fraud rate
assert np.isclose(average_precision_score(y_val, np.zeros(len(y_val))), y_val.mean())

assert not TEST_SCORED, "1996 was scored before the model was chosen!"
display(results.round(3))
print(f"✓ Step 7 | random averages: {rand_10:.1%} @10%, {rand_20:.1%} @20%")

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%
0,Random,0.065,0.096,0.213,0.062,0.096
1,Lookup: coverage + fault,0.129,0.252,0.482,0.140,0.216
2,Logistic regression,0.097,0.173,0.375,0.109,0.169


✓ Step 7 | random averages: 9.5% @10%, 19.7% @20%


we see that the logistic regression model beats our random naive baseline yet it does trail the the lookup coverage + fault. 

In [25]:
# checking to see if lookup beats lr because of the combination of base policy and fault
for d in (train_df, val_df, test_df):
    d['Coverage_Fault'] = d['BasePolicy'] + ' | ' + d['Fault']

def tiny_lr(cols):
    m = Pipeline([('onehot', OneHotEncoder(handle_unknown='ignore')),
                  ('model', LogisticRegression(class_weight='balanced', max_iter=3000))])
    m.fit(train_df[cols], y_train)
    return m.predict_proba(val_df[cols])[:, 1]

display(pd.DataFrame([
    evaluate('LR: separate',  y_val, tiny_lr(['BasePolicy', 'Fault'])),
    evaluate('LR: combined',  y_val, tiny_lr(['Coverage_Fault'])),
    evaluate('Lookup rule',   y_val, lookup_scores),
]).round(3))

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%
0,LR: separate,0.129,0.252,0.482,0.14,0.216
1,LR: combined,0.129,0.252,0.482,0.14,0.216
2,Lookup rule,0.129,0.252,0.482,0.14,0.216


In [26]:
# check 2: is there overfitting to 1994 in the full model
display(pd.DataFrame([
    evaluate('Full LR on 1994 (train)', y_train, logreg.predict_proba(X_train)[:, 1]),
    evaluate('Full LR on 1995 (val)',   y_val,   lr_scores),
]).round(3))

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%
0,Full LR on 1994 (train),0.270,0.447,0.702,0.234,0.350
1,Full LR on 1995 (val),0.097,0.173,0.375,0.109,0.169


In [27]:
# check 3 : is regularization helping? (should be a small improvement)
from sklearn.base import clone

rows = []
for C in [1.0, 0.3, 0.1, 0.03, 0.01, 0.003]:
    m = clone(logreg).set_params(model__C=C)
    m.fit(X_train, y_train)
    rows.append(evaluate(f'Full LR, C={C}', y_val, m.predict_proba(X_val)[:, 1]))
rows.append(evaluate('Lookup rule', y_val, lookup_scores))
display(pd.DataFrame(rows).round(3))

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%
0,"Full LR, C=1.0",0.097,0.173,0.375,0.109,0.169
1,"Full LR, C=0.3",0.098,0.169,0.365,0.106,0.164
2,"Full LR, C=0.1",0.099,0.156,0.359,0.104,0.161
3,"Full LR, C=0.03",0.101,0.163,0.359,0.104,0.161
4,"Full LR, C=0.01",0.104,0.166,0.352,0.102,0.158
5,"Full LR, C=0.003",0.108,0.183,0.395,0.115,0.178
6,Lookup rule,0.129,0.252,0.482,0.140,0.216


In [28]:
# check 4: which feature groups are helping and hurting
def fit_lr(num, cat):
    prep = ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                          ('scale', StandardScaler())]), num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
    m = Pipeline([('prep', prep), ('model', LogisticRegression(
        class_weight='balanced', max_iter=3000, random_state=42))])
    m.fit(train_df[num + cat], y_train)
    return m.predict_proba(val_df[num + cat])[:, 1]

core      = ['BasePolicy', 'Fault']
date_cols = ['Month', 'WeekOfMonth', 'DayOfWeek',
             'DayOfWeekClaimed', 'MonthClaimed', 'WeekOfMonthClaimed']
other_cat = [c for c in categorical_cols if c not in core + date_cols]

rows = []
for name, num, cat in [
    ('Core (BasePolicy + Fault)',   [],           core),
    ('Core + numeric',              numeric_cols, core),
    ('Core + other categories',     [],           core + other_cat),
    ('Core + dates',                [],           core + date_cols),
    ('Core + numeric + other cats', numeric_cols, core + other_cat),
    ('Everything (current model)',  numeric_cols, core + other_cat + date_cols),
]:
    rows.append(evaluate(name, y_val, fit_lr(num, cat)))
display(pd.DataFrame(rows).round(3))

,Approach,PR-AUC,Fraud caught @ top 10%,Fraud caught @ top 20%,Precision @ top 20%,F1 @ top 20%
0,Core (BasePolicy + Fault),0.129,0.252,0.482,0.140,0.216
1,Core + numeric,0.140,0.256,0.515,0.149,0.231
2,Core + other categories,0.146,0.289,0.538,0.156,0.242
3,Core + dates,0.093,0.163,0.326,0.094,0.146
4,Core + numeric + other cats,0.143,0.292,0.535,0.155,0.240
5,Everything (current model),0.097,0.173,0.375,0.109,0.169


it seems the date features are the main reason why are scores are dropping. 

Why dates hurt: six date columns become roughly 50 one-hot columns, and 1994 has only a few hundred fraud cases spread across them. The model learns things like “Tuesdays in March were risky in 1994”, which is coincidence that doesn’t repeat in 1995. There’s also a reason to distrust them on principle. The milestone notes that fraud labels partly reflect when investigations happened, so any month pattern may be about investigation timing rather than fraud.